In [1]:
import math
from typing import Callable
from statistics import NormalDist

import numpy as np


def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr


def s_cuad_update(s_cuad_prev: float, prom_prev: float, prom_curr: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return 0.0
    else:
        return (1 - (1/(n_curr - 1))) * s_cuad_prev + n_curr * ((prom_curr - prom_prev)**2)

def calculo_z_standard(alpha: float):
    return NormalDist().inv_cdf(1 - alpha / 2)


def intervalo(prom: float, s_cuad: float, n: int, alpha: float):
    z_alpha_2 = calculo_z_standard(alpha)

    std = math.sqrt(s_cuad/n)
    izq = prom - z_alpha_2 * std
    der = prom + z_alpha_2 * std

    return izq, der

### Ejercicio 3.
Para las siguientes integrales:

1.
$$
\int_{\pi}^{2\pi} \frac{\sin(x)}{x} \ dx
$$

2.
$$
\int_{0}^{\infty} \frac{3}{3 + x^4} \ dx
$$

a. Obtener mediante simulación en computadora el valor de la integral deteniendo la simulación cuando el semiancho del intervalo de confianza del $ 95\% $ sea justo inferior a $0.001$.

b. Indicar el número de simulaciones $N_S$ necesarias en la simulación realizada para lograr la condición pedida y completar con los valores obtenidos la siguiente tabla (usando 4 decimales) (LA TABLA ESTA EN EL APUNTE)

---

#### Análisis

El semi-ancho es $L / 2$, con $L = 2 * z_{\frac{\alpha}{2}} \frac{S(n)}{\sqrt{n}}$

Como $0.001 = L / 2$ entonces $L = 0.002$, por lo que:

$$
\begin{array}{rcl}

    L & = & 2 * z_{\frac{\alpha}{2}} \frac{S(n)}{\sqrt{n}} \\
    0.002 & = & 2 * z_{\frac{\alpha}{2}} \frac{S(n)}{\sqrt{n}} \\
    \frac{0.002}{2 * z_{\frac{\alpha}{2}}} & = & \frac{S(n)}{\sqrt{n}}

\end{array}
$$

<br><br>

Lo cual implica que:
$$
\begin{array}{rcl}

    \frac{S(n)}{\sqrt{n}} & \le & \frac{0.001}{z_{\frac{\alpha}{2}}} \\ \\
    \frac{S(n)}{\sqrt{n}} & \le & \frac{0.001}{z_{\frac{\alpha}{2}}}

\end{array}
$$

Ademas, como se nos basamos en la Teoría central del limite, tomamos $n \gt 100$.

Por lo cual utilizaría Monte Carlo mientras que se cumpla la siguiente condición: $n \le 100 \vee \frac{S(n)}{\sqrt{n}} \ge \frac{0.001}{z_{\frac{\alpha}{2}}}$

In [2]:
def punto_i(u: float):
    # h(u) = g(u * (b - a) + a) * (b - a)

    def g(x: float):
        return math.sin(x) / x

    pi = math.pi
    return g(u * pi + pi) * pi

def punto_ii(u: float):
    def g(x: float):
        return 3 / (3 + x**4)

    return g((1/u) - 1) / (u**2)


def simulaciones(fun: Callable[[float], float], n_min: int, l: float, alpha: float):
    # l = semi-ancho = L / 2

    z_alpha_2 = calculo_z_standard(alpha)
    cota = l / z_alpha_2

    promedio = fun(np.random.random())
    s_cuad, n = 0, 1

    while n <= n_min or math.sqrt(s_cuad / n) >= cota:
        X = fun(np.random.random())
        n += 1

        promedio_prev = promedio
        promedio = prom_update(promedio, X, n)
        s_cuad = s_cuad_update(s_cuad, promedio_prev, promedio, n)

    i_d, i_i = intervalo(promedio, s_cuad, n, alpha)
    print(
        f"{fun.__name__:>9} | {n:9d} | {f"[{i_d:.4f}, {i_i:.4f}]":>18} | {s_cuad:10.4f} | {promedio:10.4f}")

n_min = 100
l = 0.001
alpha = 0.05 # I.C 95%, ya que I.C = 1 - alpha

print(f"{"Punto":>9} | {"N° Sim":>9} | {"Intervalo obtenido":>18} | {"S_Cuadrado":>10} | {"Media":>10}")
simulaciones(punto_i, n_min, l, alpha)
simulaciones(punto_ii, n_min, l, alpha)

    Punto |    N° Sim | Intervalo obtenido | S_Cuadrado |      Media
  punto_i |    170800 | [-0.4339, -0.4319] |     0.0445 |    -0.4329
 punto_ii |   3660754 |   [1.4612, 1.4632] |     0.9530 |     1.4622
